In [87]:
import plotly.express as px
import pandas as pd
import os
import tarfile
import urllib

pd.options.plotting.backend = 'plotly'
DOWNLOAD_ROOT = " http s : / / raw . gi thubuse rcontent . com/ageron / hands on-ml /master/ ".replace(' ', '')

HOUSING_PATH = os.path.join('datasets', 'housing')

HOUSING_URL = DOWNLOAD_ROOT + " data sets/housing/hous ing . tg z ".replace(' ', '')

# info fetching:
def fetch_housing_data(housing_url=HOUSING_URL, housing_path=HOUSING_PATH):
    if not os.path.isdir(housing_path):
        os.makedirs(housing_path)
    tgz_path = os.path.join(housing_path, "housing.tgz")
    urllib.request.urlretrieve(housing_url, tgz_path)
    housing_tgz = tarfile.open(tgz_path)
    housing_tgz.extractall(path=housing_path)
    housing_tgz.close()


def load_housing_data(housing_path=HOUSING_PATH):
    csv_path = os.path.join(housing_path, " hous ing . csv ".replace(' ', ''))
    return pd.read_csv(csv_path)

In [70]:
fetch_housing_data()
housing = load_housing_data()
housing.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [71]:
housing.columns

Index(['longitude', 'latitude', 'housing_median_age', 'total_rooms',
       'total_bedrooms', 'population', 'households', 'median_income',
       'median_house_value', 'ocean_proximity'],
      dtype='object')

In [72]:
housing.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  float64
 9   ocean_proximity     20640 non-null  object 
dtypes: float64(9), object(1)
memory usage: 1.6+ MB


In [73]:
a = housing['ocean_proximity'].value_counts()

b = housing['ocean_proximity'].describe()

print(a, b, sep='\n\n\n')

<1H OCEAN     9136
INLAND        6551
NEAR OCEAN    2658
NEAR BAY      2290
ISLAND           5
Name: ocean_proximity, dtype: int64


count         20640
unique            5
top       <1H OCEAN
freq           9136
Name: ocean_proximity, dtype: object


In [74]:
housing.describe()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
count,20640.000000,20640.000000,20640.000000,20640.000000,20433.000000,20640.000000,20640.000000,20640.000000,20640.000000
mean,-119.569704,35.631861,28.639486,2635.763081,537.870553,1425.476744,499.539680,3.870671,206855.816909
std,2.003532,2.135952,12.585558,2181.615252,421.385070,1132.462122,382.329753,1.899822,115395.615874
min,-124.350000,32.540000,1.000000,2.000000,1.000000,3.000000,1.000000,0.499900,14999.000000
25%,-121.800000,33.930000,18.000000,1447.750000,296.000000,787.000000,280.000000,2.563400,119600.000000
50%,-118.490000,34.260000,29.000000,2127.000000,435.000000,1166.000000,409.000000,3.534800,179700.000000
75%,-118.010000,37.710000,37.000000,3148.000000,647.000000,1725.000000,605.000000,4.743250,264725.000000
max,-114.310000,41.950000,52.000000,39320.000000,6445.000000,35682.000000,6082.000000,15.000100,500001.000000


In [80]:
# import plotly.graph_objects as go
# from plotly.subplots import make_subplots
# from math import floor, ceil, sqrt
#
# n = len(housing.columns)
# fig = make_subplots(rows=ceil(sqrt(n)), cols=floor(n / sqrt(n)))
# for index, column in enumerate(housing):
#     trace = go.Histogram(
#         x=housing[column]
#     )
#     fig.append_trace()

# Test sample

In [83]:
import numpy as np
def split_train_test(data, test_ratio):
    np.random.seed(42)
    shuffled_indices = np.random.permutation(len(data))
    test_set_size = int(len(data) * test_ratio)
    test_indices = shuffled_indices[:test_set_size]
    train_indices = shuffled_indices[test_set_size:]
    return data.iloc[train_indices], data.iloc[test_indices]

train_set, test_set = split_train_test(housing, 0.2)

print(f'{len(train_set)} train set + {len(test_set)} test set')

16512 train set + 4128 test set


In [97]:
import sklearn as sk
import sklearn.model_selection as ms

# The same
train_set, test_set = ms.train_test_split(housing, test_size=0.2,
                                                          random_state=41)
print(f'{len(train_set)} train set + {len(test_set)} test set')

16512 train set + 4128 test set


In [98]:
# housing['median_income'].hist()

### Discretize 'madian_income' category

In [99]:
housing['income_category'] = np.ceil(housing['median_income'] / 1.5)

housing['income_category'].where(housing['income_category'] < 5, 5.0,
                                 inplace=True)

px.histogram(housing.income_category).update_layout(bargap=0.2)

In [ ]:
split = ms.StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
